In [ ]:
from nas.learning_utils import HardwareMetricEstimator
from _generated_model.J2WLVC import _model__layers

print(_model__layers())
estimator = HardwareMetricEstimator(applied_hardware="myriadvpu_openvino2019r2", hardware_metrics=["latency"]) # , "energy"
hardware_estimated_result = estimator.estimate(_model__layers())


In [ ]:
# def my_import(name):
#     components = name.split('.')
#     print(components)
#     mod = __import__(components[0])
#     for comp in components[1:]:
#         mod = getattr(mod, comp)
#     return mod

# path = "_generated_model/XIUEBY.py"
# module_name = path.replace('/', '.')[-26:-3] +"._model"
# _model__layers = my_import(module_name)
import torch
from _generated_model.ERC8JL import _model


from nni.retiarii.converter import convert_to_graph
from nni.retiarii.converter.graph_gen import GraphConverterWithShape
dummy_input = (1, 1, 33)
model = _model()
print(model)
script_module = torch.jit.script(model)
print(script_module)
base_model_ir = convert_to_graph(script_module, model, converter=GraphConverterWithShape(), dummy_input=torch.randn(*dummy_input))
        


In [1]:
import random

import nni
import torch
import torch.nn.functional as F
# remember to import nni.retiarii.nn.pytorch as nn, instead of torch.nn as nn
import nni.retiarii.nn.pytorch as nn
import nni.retiarii.strategy as strategy
from nni.retiarii import model_wrapper
from nni.retiarii.evaluator import FunctionalEvaluator
from nni.retiarii.experiment.pytorch import RetiariiExeConfig, RetiariiExperiment, debug_mutated_model
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=3, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


@model_wrapper
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        # LayerChoice is used to select a layer between Conv2d and DwConv.
        self.conv2 = nn.LayerChoice([
            nn.Conv2d(32, 64, 3, 1),
            DepthwiseSeparableConv(32, 64)
        ])
        # ValueChoice is used to select a dropout rate.
        # ValueChoice can be used as parameter of modules wrapped in `nni.retiarii.nn.pytorch`
        # or customized modules wrapped with `@basic_unit`.
        self.dropout1 = nn.Dropout(nn.ValueChoice([0.25, 0.5, 0.75]))
        self.dropout2 = nn.Dropout(0.5)
        feature = nn.ValueChoice([64, 128, 256])
        # Same value choice can be used multiple times
        self.fc1 = nn.Linear(9216, feature)
        self.fc2 = nn.Linear(feature, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(self.conv2(x), 2)
        x = torch.flatten(self.dropout1(x), 1)
        x = self.fc2(self.dropout2(F.relu(self.fc1(x))))
        return x


def train_epoch(model, device, train_loader, optimizer, epoch):
    loss_fn = torch.nn.CrossEntropyLoss()
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))


def test_epoch(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    accuracy = 100. * correct / len(test_loader.dataset)

    print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(test_loader.dataset), accuracy))

    return accuracy


def evaluate_model(model_cls):
    # "model_cls" is a class, need to instantiate
    model = model_cls()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    transf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_loader = DataLoader(MNIST('data/mnist', download=True, transform=transf), batch_size=64, shuffle=True)
    test_loader = DataLoader(MNIST('data/mnist', download=True, train=False, transform=transf), batch_size=64)

    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

    for epoch in range(3):
        # train the model for one epoch
        train_epoch(model, device, train_loader, optimizer, epoch)
        # test the model for one epoch
        accuracy = test_epoch(model, device, test_loader)
        # call report intermediate result. Result can be float or dict
        nni.report_intermediate_result(accuracy)

    # report final test result
    nni.report_final_result(accuracy)


if __name__ == '__main__':
    base_model = Net()

    search_strategy = strategy.Random()
    model_evaluator = FunctionalEvaluator(evaluate_model)

    exp = RetiariiExperiment(base_model, model_evaluator, [], search_strategy)

    exp_config = RetiariiExeConfig('local')
    exp_config.experiment_name = 'mnist_search'
    exp_config.trial_concurrency = 2
    exp_config.max_trial_number = 20
    exp_config.training_service.use_active_gpu = False
    export_formatter = 'dict'

    # uncomment this for graph-based execution engine
    # exp_config.execution_engine = 'base'
    # export_formatter = 'code'

    exp.run(exp_config, 8081 + random.randint(0, 100))
    print('Final model:')
    for model_code in exp.export_top_models(formatter=export_formatter):
        print(model_code)

/home/locla/EnergyNAS/nni/common/serializer.py:564: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  return super().__new__(cls, name, bases, dct)


[2024-09-07 23:47:48] INFO (hyperopt.utils/MainThread) Failed to load dill, try installing dill via "pip install dill" for enhanced pickling support.
[2024-09-07 23:47:48] INFO (hyperopt.fmin/MainThread) Failed to load dill, try installing dill via "pip install dill" for enhanced pickling support.
[2024-09-07 23:47:49] INFO (root/MainThread) PyTorch Lightning is not installed.
[2024-09-07 23:47:49] INFO (root/MainThread) Tensorflow is not installed.
[2024-09-07 23:47:49] INFO (nni.experiment/MainThread) Creating experiment, Experiment ID: is7wp4cv
[2024-09-07 23:47:49] INFO (nni.experiment/MainThread) Connecting IPC pipe...
[2024-09-07 23:47:49] INFO (nni.experiment/MainThread) Starting web server...
[2024-09-07 23:47:50] INFO (nni.experiment/MainThread) Setting up...
[2024-09-07 23:47:50] INFO (nni.runtime.msg_dispatcher_base/Thread-6) Dispatcher started
[2024-09-07 23:47:50] INFO (nni.retiarii.experiment.pytorch/MainThread) Web UI URLs: http://127.0.0.1:8084 http://10.42.0.1:8084 htt

: 

In [10]:
import onnx 
from onnx import shape_inference
        
model = onnx.load("_generated_model/0D4WZT.onnx")
inferred_model = shape_inference.infer_shapes(model)
for node in inferred_model.graph.node:
    print(node.name)
    if node.name == "Identity_1":
        print(node)

/_first_layer/MatMul
/_first_layer/Add
/_resnetblocks/_blocks__0/Constant
/_resnetblocks/_blocks__0/Squeeze
/_resnetblocks/_blocks__0/_linear_1/Gemm
/_resnetblocks/_blocks__0/_relu_1/Relu
/_resnetblocks/_blocks__0/_linear_2/Gemm
/_resnetblocks/_blocks__0/Add
/_resnetblocks/_blocks__1/_linear_1/Gemm
/_resnetblocks/_blocks__1/_relu_1/Relu
/_resnetblocks/_blocks__1/_linear_2/Gemm
/_resnetblocks/_blocks__1/Add
/_resnetblocks/_blocks__2/_linear_1/Gemm
/_resnetblocks/_blocks__2/_relu_1/Relu
/_resnetblocks/_blocks__2/_linear_2/Gemm
/_resnetblocks/_blocks__2/Add
/_head/_relu/Relu
/_head/_linear/Gemm


: 